# Using LLMs in Practice: Private APIs & Open-Source Models

This notebook teaches you how to use LLMs in your projects, covering:

1. **Private/Commercial APIs**: OpenAI, Anthropic Claude, Google Gemini, Mistral
2. **Open-Source Models**: LLaMA, Qwen, Mistral (local), Phi
3. **Unified Interfaces**: LiteLLM, LangChain
4. **Best Practices**: API keys, rate limiting, cost management

---

## Setup

Install required packages:

In [ ]:
# Install packages (uncomment as needed)
# !pip install openai anthropic google-generativeai mistralai
# !pip install transformers torch accelerate
# !pip install litellm langchain langchain-openai langchain-anthropic

### API Key Management

**Never hardcode API keys!** Use environment variables or a `.env` file.

In [ ]:
import os
from dotenv import load_dotenv

# Load from .env file (create one with your keys)
load_dotenv()

# Or set directly (for demo only - don't commit this!)
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["GOOGLE_API_KEY"] = "..."
# os.environ["MISTRAL_API_KEY"] = "..."

print("API keys loaded:", {
    "OpenAI": "OPENAI_API_KEY" in os.environ,
    "Anthropic": "ANTHROPIC_API_KEY" in os.environ,
    "Google": "GOOGLE_API_KEY" in os.environ,
    "Mistral": "MISTRAL_API_KEY" in os.environ,
})

---

# Part 1: Private/Commercial LLM APIs

These are hosted services where you pay per token. No GPU required!

## 1.1 OpenAI (GPT-4, GPT-3.5)

The most popular LLM API. Models include:
- `gpt-4o` - Latest multimodal model
- `gpt-4-turbo` - Fast GPT-4
- `gpt-3.5-turbo` - Cheaper, faster, good for simple tasks

**Pricing** (as of 2024):
- GPT-4o: $5/1M input tokens, $15/1M output tokens
- GPT-3.5-turbo: $0.50/1M input, $1.50/1M output

In [ ]:
from openai import OpenAI

# Initialize client
client = OpenAI()  # Uses OPENAI_API_KEY env var

# Simple completion
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain transformers in one sentence."}
    ],
    max_tokens=100,
    temperature=0.7,
)

print(response.choices[0].message.content)

In [ ]:
# Streaming response (for real-time output)
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Write a haiku about coding."}],
    stream=True,
)

print("Streaming response:")
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

In [ ]:
# Function calling (structured output)
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather in a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "City name"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
                },
                "required": ["location"]
            }
        }
    }
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What's the weather in Tokyo?"}],
    tools=tools,
    tool_choice="auto",
)

print("Function call:", response.choices[0].message.tool_calls)

## 1.2 Anthropic Claude

Known for safety, long context (200K tokens), and strong reasoning.

Models:
- `claude-3-5-sonnet-20241022` - Best balance of speed/quality
- `claude-3-opus-20240229` - Most capable
- `claude-3-haiku-20240307` - Fastest, cheapest

**Pricing**:
- Sonnet: $3/1M input, $15/1M output
- Haiku: $0.25/1M input, $1.25/1M output

In [ ]:
import anthropic

# Initialize client
client = anthropic.Anthropic()  # Uses ANTHROPIC_API_KEY env var

# Simple completion
message = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "Explain attention mechanism in transformers."}
    ]
)

print(message.content[0].text)

In [ ]:
# With system prompt
message = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=1024,
    system="You are a Python expert. Provide concise code examples.",
    messages=[
        {"role": "user", "content": "Show me how to implement a simple LRU cache."}
    ]
)

print(message.content[0].text)

In [ ]:
# Streaming with Claude
with client.messages.stream(
    model="claude-3-haiku-20240307",
    max_tokens=256,
    messages=[{"role": "user", "content": "Write a limerick about machine learning."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
print()

## 1.3 Google Gemini

Google's multimodal models with long context (up to 2M tokens!).

Models:
- `gemini-1.5-pro` - Most capable, 2M context
- `gemini-1.5-flash` - Fast and efficient
- `gemini-2.0-flash-exp` - Latest experimental

In [ ]:
import google.generativeai as genai

# Configure API key
genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))

# Initialize model
model = genai.GenerativeModel("gemini-1.5-flash")

# Simple generation
response = model.generate_content("Explain RLHF in simple terms.")
print(response.text)

In [ ]:
# Chat conversation
chat = model.start_chat(history=[])

response = chat.send_message("What is a transformer?")
print("Q1:", response.text[:200], "...\n")

response = chat.send_message("How does attention work in it?")
print("Q2:", response.text[:200], "...")

## 1.4 Mistral AI

European AI company with strong open-weight and API models.

Models:
- `mistral-large-latest` - Most capable
- `mistral-medium-latest` - Balanced
- `mistral-small-latest` - Fast and cheap
- `codestral-latest` - Optimized for code

In [ ]:
from mistralai import Mistral

# Initialize client
client = Mistral(api_key=os.environ.get("MISTRAL_API_KEY"))

# Simple completion
response = client.chat.complete(
    model="mistral-small-latest",
    messages=[
        {"role": "user", "content": "What makes Mistral models unique?"}
    ]
)

print(response.choices[0].message.content)

## 1.5 Comparison: Which API to Choose?

| Provider | Best For | Context | Strengths |
|----------|----------|---------|------------|
| **OpenAI** | General use, ecosystem | 128K | Best tooling, function calling |
| **Anthropic** | Safety, long docs | 200K | Reasoning, following instructions |
| **Google** | Multimodal, very long context | 2M | Video/audio, massive context |
| **Mistral** | Europe, open-weight | 32K | Price, EU data residency |

---

# Part 2: Open-Source LLMs (Local Inference)

Run models on your own hardware. Requires GPU for good performance.

## 2.1 Using Hugging Face Transformers

The standard library for loading and running open-source models.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Check available device
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

### Load a Small Model (for CPU/limited GPU)

Microsoft's Phi models are great for learning - small but capable.

In [ ]:
# Phi-2 (2.7B parameters) - runs on most hardware
model_name = "microsoft/phi-2"

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device != "cpu" else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)
print("Model loaded!")

In [ ]:
# Generate text
prompt = "Explain what a neural network is in simple terms:"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

### Using Pipeline (Simpler API)

In [ ]:
# Create a text generation pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

# Generate
result = generator(
    "The key innovation of transformers is",
    max_new_tokens=50,
    temperature=0.7,
    do_sample=True,
)

print(result[0]["generated_text"])

## 2.2 Meta LLaMA Models

Meta's open-weight models. Requires accepting license on Hugging Face.

Models:
- `meta-llama/Llama-3.2-1B` - Tiny, fast
- `meta-llama/Llama-3.2-3B` - Small but capable
- `meta-llama/Llama-3.1-8B-Instruct` - Great balance
- `meta-llama/Llama-3.1-70B-Instruct` - Very capable (needs big GPU)

In [ ]:
# LLaMA 3.2 1B - Small enough for most hardware
# Note: You need to accept the license at https://huggingface.co/meta-llama/Llama-3.2-1B
# and login with: huggingface-cli login

llama_model_name = "meta-llama/Llama-3.2-1B-Instruct"

try:
    llama_tokenizer = AutoTokenizer.from_pretrained(llama_model_name)
    llama_model = AutoModelForCausalLM.from_pretrained(
        llama_model_name,
        torch_dtype=torch.float16 if device != "cpu" else torch.float32,
        device_map="auto",
    )
    print(f"Loaded {llama_model_name}")
except Exception as e:
    print(f"Could not load LLaMA: {e}")
    print("Make sure you've accepted the license and logged in with huggingface-cli")

In [ ]:
# LLaMA chat format
def format_llama_prompt(system_prompt, user_message):
    """Format prompt for LLaMA 3 Instruct models."""
    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>

{user_message}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

# Generate with LLaMA
if 'llama_model' in dir():
    prompt = format_llama_prompt(
        "You are a helpful AI assistant.",
        "What is the difference between GPT and LLaMA?"
    )
    
    inputs = llama_tokenizer(prompt, return_tensors="pt").to(llama_model.device)
    outputs = llama_model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
    )
    
    response = llama_tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the assistant's response
    print(response.split("assistant")[-1].strip())

## 2.3 Qwen Models (Alibaba)

Strong multilingual models from Alibaba, especially good for Chinese.

Models:
- `Qwen/Qwen2.5-0.5B-Instruct` - Tiny
- `Qwen/Qwen2.5-1.5B-Instruct` - Small
- `Qwen/Qwen2.5-7B-Instruct` - Great balance
- `Qwen/Qwen2.5-72B-Instruct` - Very capable

In [ ]:
# Qwen 2.5 0.5B - Very small, runs anywhere
qwen_model_name = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading {qwen_model_name}...")
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    torch_dtype=torch.float16 if device != "cpu" else torch.float32,
    device_map="auto",
)
print("Qwen loaded!")

In [ ]:
# Qwen uses ChatML format
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain what makes Qwen models special."}
]

# Apply chat template
text = qwen_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = qwen_tokenizer(text, return_tensors="pt").to(qwen_model.device)
outputs = qwen_model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.7,
    do_sample=True,
)

response = qwen_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response.split("assistant")[-1].strip())

## 2.4 Mistral Models (Open-Weight)

Mistral also releases open-weight versions of their models.

- `mistralai/Mistral-7B-Instruct-v0.3` - Great 7B model
- `mistralai/Mixtral-8x7B-Instruct-v0.1` - MoE model (needs more VRAM)

In [ ]:
# Mistral 7B - needs ~14GB VRAM in fp16
# Uncomment if you have enough GPU memory

# mistral_model_name = "mistralai/Mistral-7B-Instruct-v0.3"
# 
# mistral_tokenizer = AutoTokenizer.from_pretrained(mistral_model_name)
# mistral_model = AutoModelForCausalLM.from_pretrained(
#     mistral_model_name,
#     torch_dtype=torch.float16,
#     device_map="auto",
# )

print("Mistral 7B requires ~14GB VRAM. Uncomment above if you have it.")

## 2.5 Quantized Models (Run Large Models on Less VRAM)

Quantization reduces model precision (fp16 → int8 → int4) to use less memory.

In [ ]:
# Using bitsandbytes for 4-bit quantization
# !pip install bitsandbytes

from transformers import BitsAndBytesConfig

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load a larger model in 4-bit (e.g., 7B model uses ~4GB instead of ~14GB)
# model = AutoModelForCausalLM.from_pretrained(
#     "mistralai/Mistral-7B-Instruct-v0.3",
#     quantization_config=bnb_config,
#     device_map="auto",
# )

print("4-bit quantization can reduce VRAM by ~4x!")
print("7B model: 14GB → ~4GB")
print("70B model: 140GB → ~35GB")

---

# Part 3: Unified Interfaces

Use the same code for any LLM provider.

## 3.1 LiteLLM

One interface for 100+ LLM providers. Uses OpenAI-compatible format.

In [ ]:
from litellm import completion

# Same code works for any provider!
def ask_llm(model, question):
    """Query any LLM with the same interface."""
    response = completion(
        model=model,
        messages=[{"role": "user", "content": question}],
        max_tokens=100,
    )
    return response.choices[0].message.content

# Try different providers (uncomment ones you have API keys for)
question = "What is 2+2? Answer in one word."

# OpenAI
# print("OpenAI:", ask_llm("gpt-4o-mini", question))

# Anthropic
# print("Claude:", ask_llm("claude-3-haiku-20240307", question))

# Google
# print("Gemini:", ask_llm("gemini/gemini-1.5-flash", question))

# Mistral
# print("Mistral:", ask_llm("mistral/mistral-small-latest", question))

print("LiteLLM provides a unified interface for all providers!")

## 3.2 LangChain

Framework for building LLM applications with chains, agents, and tools.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

# Initialize models
# openai_llm = ChatOpenAI(model="gpt-4o-mini")
# claude_llm = ChatAnthropic(model="claude-3-haiku-20240307")

# Same interface for both
# messages = [
#     SystemMessage(content="You are a helpful assistant."),
#     HumanMessage(content="What is the capital of France?")
# ]

# response = openai_llm.invoke(messages)
# print(response.content)

print("LangChain provides chains, agents, and tools for LLM apps!")

In [ ]:
# LangChain with local models via Hugging Face
from langchain_huggingface import HuggingFacePipeline

# Use the Qwen model we loaded earlier
if 'qwen_model' in dir():
    local_pipeline = pipeline(
        "text-generation",
        model=qwen_model,
        tokenizer=qwen_tokenizer,
        max_new_tokens=100,
    )
    
    local_llm = HuggingFacePipeline(pipeline=local_pipeline)
    
    # Now use it like any other LangChain model
    response = local_llm.invoke("What is machine learning?")
    print(response)

---

# Part 4: Best Practices

## 4.1 API Key Security

**Never commit API keys to git!**

In [ ]:
# Good: Use environment variables
import os
api_key = os.environ.get("OPENAI_API_KEY")

# Good: Use .env file with python-dotenv
from dotenv import load_dotenv
load_dotenv()  # Loads from .env file

# Good: Use secrets manager in production
# from aws_secretsmanager import get_secret
# api_key = get_secret("openai-api-key")

# BAD: Never do this!
# api_key = "sk-abc123..."  # NEVER hardcode keys!

print("Always use environment variables or secrets managers!")

## 4.2 Rate Limiting and Retries

In [ ]:
import time
from tenacity import retry, stop_after_attempt, wait_exponential

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=4, max=60)
)
def call_llm_with_retry(client, messages):
    """Call LLM with automatic retry on rate limits."""
    return client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )

# Simple rate limiter
class RateLimiter:
    def __init__(self, calls_per_minute):
        self.calls_per_minute = calls_per_minute
        self.calls = []
    
    def wait_if_needed(self):
        now = time.time()
        self.calls = [t for t in self.calls if now - t < 60]
        if len(self.calls) >= self.calls_per_minute:
            sleep_time = 60 - (now - self.calls[0])
            time.sleep(sleep_time)
        self.calls.append(time.time())

print("Use retries and rate limiting for production!")

## 4.3 Cost Management

In [ ]:
import tiktoken

def estimate_cost(text, model="gpt-4o-mini"):
    """Estimate cost before making API call."""
    # Pricing per 1M tokens (as of 2024)
    pricing = {
        "gpt-4o": {"input": 5.0, "output": 15.0},
        "gpt-4o-mini": {"input": 0.15, "output": 0.60},
        "gpt-3.5-turbo": {"input": 0.50, "output": 1.50},
        "claude-3-5-sonnet": {"input": 3.0, "output": 15.0},
        "claude-3-haiku": {"input": 0.25, "output": 1.25},
    }
    
    # Count tokens
    enc = tiktoken.get_encoding("cl100k_base")
    tokens = len(enc.encode(text))
    
    # Estimate (assuming output is similar length to input)
    if model in pricing:
        input_cost = (tokens / 1_000_000) * pricing[model]["input"]
        output_cost = (tokens / 1_000_000) * pricing[model]["output"]
        return {
            "tokens": tokens,
            "estimated_input_cost": f"${input_cost:.6f}",
            "estimated_output_cost": f"${output_cost:.6f}",
        }
    return {"tokens": tokens, "model_not_found": model}

# Example
text = "Explain quantum computing in detail." * 100
print(estimate_cost(text, "gpt-4o-mini"))

## 4.4 Model Selection Guide

| Use Case | Recommended Model | Why |
|----------|-------------------|-----|
| **Quick prototyping** | GPT-4o-mini, Claude Haiku | Fast, cheap |
| **Production chat** | GPT-4o, Claude Sonnet | Good balance |
| **Complex reasoning** | Claude Opus, o1 | Best reasoning |
| **Long documents** | Gemini 1.5 Pro, Claude | Long context |
| **Code generation** | Claude Sonnet, GPT-4o | Best at code |
| **Local/private** | LLaMA 3.1 8B, Qwen 7B | No API needed |
| **Edge/mobile** | Phi-2, Qwen 0.5B | Tiny models |

---

# Summary

## Private APIs
- **OpenAI**: Best ecosystem, function calling
- **Anthropic**: Safety, long context, reasoning
- **Google**: Multimodal, 2M context
- **Mistral**: EU data residency, good value

## Open-Source Models
- **LLaMA**: Meta's flagship, great community
- **Qwen**: Strong multilingual, many sizes
- **Mistral**: Efficient, MoE variants
- **Phi**: Tiny but capable

## Unified Interfaces
- **LiteLLM**: One API for all providers
- **LangChain**: Build LLM applications

## Best Practices
- Never hardcode API keys
- Use retries and rate limiting
- Estimate costs before large batches
- Choose model based on use case

---

# Exercises

1. **Compare providers**: Ask the same question to OpenAI, Claude, and Gemini. Compare response quality and latency.

2. **Local inference**: Load Qwen 0.5B and measure tokens/second on your hardware.

3. **Cost optimization**: Given a dataset of 10,000 prompts, calculate the cost difference between GPT-4o and GPT-4o-mini.

4. **Build a chatbot**: Use LangChain to build a simple chatbot with conversation memory.

5. **Quantization**: Compare the output quality of a 7B model in fp16 vs 4-bit quantization.